# LECTURE 18 — DEMONSTRATION
### MODULE 7: UNSUPERVISED LEARNING & DIMENSIONALITY REDUCTION

**DEMONSTRATION** — Principal Component Analysis (PCA) & Dimensionality Reduction

*WAIFEM ML Facilitation Programme 2026 — Compatible with Google Colab & VS Code*

## ── SECTION 1: IMPORTS

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 11})

try:
    import google.colab          # noqa: F401
    print("Environment : Google Colab")
except ImportError:
    print("Environment : Local (VS Code / terminal)")

print("Libraries loaded successfully.\n")

## ── SECTION 2: DATA GENERATION — 12 CORRELATED INDICATORS

In [ ]:
COUNTRIES = [
    'Nigeria',        'South Africa',   'Egypt',          'Kenya',
    'Ethiopia',       'Ghana',          'Tanzania',       "Cote d'Ivoire",
    'Cameroon',       'Angola',         'Uganda',         'Mozambique',
    'Zambia',         'Zimbabwe',       'Senegal',        'Mali',
    'Burkina Faso',   'Guinea',         'Niger',          'Rwanda',
    'Benin',          'Burundi',        'Sierra Leone',   'Togo',
    'Libya',          'Tunisia',        'Algeria',        'Morocco',
    'Sudan',          'Madagascar',     'Botswana',       'Namibia',
    'Mauritius',      'Malawi',         'DR Congo',       'Congo Republic',
    'Gabon',          'Eq. Guinea',     'Eswatini',       'Lesotho',
    'Djibouti',       'Eritrea',        'Somalia',        'Chad',
    'Cabo Verde',
]
N = len(COUNTRIES)


def generate_resilience_data(countries: list, seed: int = 42) -> pd.DataFrame:
    """
    Generate 12 macroeconomic indicators with genuine latent structure
    suitable for PCA.  Three underlying dimensions drive the data:
      F1 — Macroeconomic stability  (inflation−, fiscal−, debt−, CA−, reserves+)
      F2 — Development              (GDP pc+, HDI+, electricity+, schooling+)
      F3 — Financial depth          (credit+, remittances+, life expectancy+)
    """
    rng = np.random.default_rng(seed)

    # Latent factor scores for each country
    F1 = rng.normal(0, 1, N)    # macro stability
    F2 = rng.normal(0, 1, N)    # development
    F3 = rng.normal(0, 1, N)    # financial depth

    e = lambda s: rng.normal(0, s, N)  # idiosyncratic noise

    # Construct indicators from factors (loadings) + noise
    gdp_pc          = 1800 + 900 * F2  + 300 * F1  + e(200)
    gdp_growth      =  4.0 + 0.8 * F2  + 0.4 * F1  + e(0.8)
    inflation       = 10.0 - 2.5 * F1  + 0.5 * F2  + e(1.5)
    fiscal_balance  = -3.0 + 1.2 * F1  - 0.3 * F2  + e(0.8)
    debt_to_gdp     = 55.0 - 8.0 * F1  + 2.0 * F2  + e(6)
    current_account = -3.0 + 1.5 * F1  + 0.5 * F2  + e(1.0)
    reserves_months =  3.5 + 1.0 * F1  + 0.4 * F3  + e(0.6)
    hdi             =  0.5 + 0.08 * F2 + 0.02 * F3 + e(0.03)
    electricity     = 55.0 + 18.0 * F2 + 5.0 * F3  + e(8)
    school_enrol    = 75.0 + 10.0 * F2 + 3.0 * F3  + e(5)
    life_expectancy = 62.0 + 5.0 * F2  + 2.0 * F3  + e(3)
    credit_to_gdp   = 28.0 + 4.0 * F3  + 3.0 * F2  + e(5)

    # Clip to plausible ranges
    data = {
        'gdp_per_capita':    np.clip(gdp_pc,          300,  9000),
        'gdp_growth':        np.clip(gdp_growth,       -3,    12),
        'inflation':         np.clip(inflation,          1,    50),
        'fiscal_balance':    np.clip(fiscal_balance,   -12,     4),
        'debt_to_gdp':       np.clip(debt_to_gdp,       15,   120),
        'current_account':   np.clip(current_account,  -12,     6),
        'reserves_months':   np.clip(reserves_months,   0.5,    9),
        'hdi':               np.clip(hdi,               0.25,  0.85),
        'electricity_access':np.clip(electricity,       10,    99),
        'school_enrollment': np.clip(school_enrol,      40,    99),
        'life_expectancy':   np.clip(life_expectancy,   50,    78),
        'credit_to_gdp':     np.clip(credit_to_gdp,      5,    80),
    }
    return pd.DataFrame(data, index=countries)


df = generate_resilience_data(COUNTRIES)
INDICATOR_COLS = df.columns.tolist()

print(f"Dataset : {df.shape[0]} countries  x  {len(INDICATOR_COLS)} indicators")
print(df.describe().round(2).to_string())

## ── SECTION 3: STANDARDISE

In [ ]:
scaler = StandardScaler()
X_sc   = scaler.fit_transform(df.values)

print("\nFeatures standardised (mean=0, std=1).")

# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pd.DataFrame(X_sc, columns=INDICATOR_COLS).corr(),
            annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.4, square=True, ax=ax, annot_kws={'size': 7})
ax.set_title('Correlation Matrix — Macroeconomic Resilience Indicators',
             fontweight='bold')
plt.tight_layout()
plt.savefig('lecture18_demo_correlation.png', bbox_inches='tight')
plt.show()
print("Saved : lecture18_demo_correlation.png")

## ── SECTION 4: PCA — FIT

In [ ]:
pca_full = PCA(random_state=SEED)
pca_full.fit(X_sc)

evr        = pca_full.explained_variance_ratio_
cum_evr    = np.cumsum(evr)
n_pcs      = len(evr)
n_90       = int(np.argmax(cum_evr >= 0.90)) + 1

print(f"\nPCA fitted on {len(INDICATOR_COLS)} indicators")
print(f"  Components to explain 90 % variance : {n_90}")

# Variance explained table
ve_df = pd.DataFrame({
    'PC':              [f'PC{i+1}' for i in range(n_pcs)],
    'Eigenvalue':      pca_full.explained_variance_.round(3),
    'Var Explained %': (evr * 100).round(2),
    'Cumulative %':    (cum_evr * 100).round(2),
})
print(ve_df.to_string(index=False))

## ── SECTION 5: SCREE PLOT

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('PCA — Scree Plot & Cumulative Variance Explained', fontsize=12, fontweight='bold')

axes[0].bar(range(1, n_pcs + 1), evr * 100, color='steelblue', alpha=0.85)
axes[0].axhline(5, color='tomato', linestyle='--', linewidth=1.2, label='5 % threshold')
axes[0].set_title('Individual Variance Explained (%)')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained (%)')
axes[0].legend()
axes[0].grid(True, axis='y', alpha=0.3)

axes[1].plot(range(1, n_pcs + 1), cum_evr * 100, marker='o', color='seagreen', linewidth=1.8)
axes[1].axhline(90, color='tomato', linestyle='--', linewidth=1.2, label='90 % threshold')
axes[1].axvline(n_90, color='grey', linestyle=':', linewidth=1.2,
                label=f'{n_90} PCs → 90 % variance')
axes[1].set_title('Cumulative Variance Explained (%)')
axes[1].set_xlabel('Principal Component')
axes[1].set_ylabel('Cumulative Variance (%)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lecture18_demo_scree.png', bbox_inches='tight')
plt.show()
print("Saved : lecture18_demo_scree.png")

## ── SECTION 6: LOADINGS HEATMAP

In [ ]:
pca3      = PCA(n_components=3, random_state=SEED)
scores    = pca3.fit_transform(X_sc)        # (N, 3)
loadings  = pd.DataFrame(
    pca3.components_.T,
    index=INDICATOR_COLS,
    columns=[f'PC{i+1}' for i in range(3)],
)

fig, ax = plt.subplots(figsize=(7, 7))
sns.heatmap(loadings, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, square=False, ax=ax)
ax.set_title('PCA Loadings — First 3 Principal Components', fontweight='bold')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Indicator')
plt.tight_layout()
plt.savefig('lecture18_demo_loadings.png', bbox_inches='tight')
plt.show()
print("Saved : lecture18_demo_loadings.png")

print("\n── Factor Interpretation ────────────────────────────────────────────")
print(f"  PC1 ({evr[0]*100:.1f}%) — Macroeconomic Stability")
print(f"      High loading: reserves, fiscal balance, current account (positive)")
print(f"                    inflation, debt-to-GDP (negative)")
print(f"  PC2 ({evr[1]*100:.1f}%) — Development Level")
print(f"      High loading: GDP pc, HDI, electricity access, school enrollment")
print(f"  PC3 ({evr[2]*100:.1f}%) — Financial Depth")
print(f"      High loading: credit-to-GDP, life expectancy, remittances")

## ── SECTION 7: BIPLOT (PC1 vs PC2)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))

# Country scores
ax.scatter(scores[:, 0], scores[:, 1], color='steelblue', s=60, zorder=3, alpha=0.8)
for i, name in enumerate(COUNTRIES):
    ax.annotate(name, (scores[i, 0], scores[i, 1]),
                fontsize=6, ha='center', va='bottom',
                xytext=(0, 4), textcoords='offset points')

# Loading arrows
scale = 3.0
for j, feat in enumerate(INDICATOR_COLS):
    ax.annotate('', xy=(loadings.iloc[j, 0] * scale, loadings.iloc[j, 1] * scale),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='tomato', lw=1.5))
    ax.text(loadings.iloc[j, 0] * scale * 1.10,
            loadings.iloc[j, 1] * scale * 1.10,
            feat.replace('_', '\n'), fontsize=7, color='tomato', ha='center')

ax.axhline(0, color='grey', linewidth=0.6, linestyle=':')
ax.axvline(0, color='grey', linewidth=0.6, linestyle=':')
ax.set_title(f'PCA Biplot — PC1 ({evr[0]*100:.1f} %) vs PC2 ({evr[1]*100:.1f} %)',
             fontsize=11, fontweight='bold')
ax.set_xlabel(f'PC1 — Macroeconomic Stability ({evr[0]*100:.1f} % variance)')
ax.set_ylabel(f'PC2 — Development Level ({evr[1]*100:.1f} % variance)')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('lecture18_demo_biplot.png', bbox_inches='tight')
plt.show()
print("Saved : lecture18_demo_biplot.png")

## ── SECTION 8: MACROECONOMIC RESILIENCE INDEX

PC1 captures macro-stability; higher score = more resilient.

We invert the sign if PC1 correlates negatively with "good" indicators.

In [ ]:
pc1_sign = np.sign(loadings.loc['reserves_months', 'PC1'])   # reserves should be positive
resilience_raw = scores[:, 0] * pc1_sign

# Rescale to 0–100
ri_min, ri_max = resilience_raw.min(), resilience_raw.max()
resilience_idx = 100 * (resilience_raw - ri_min) / (ri_max - ri_min)

df['resilience_index'] = resilience_idx
ranking = df[['resilience_index']].sort_values('resilience_index', ascending=False)
ranking['rank'] = range(1, N + 1)

print("\n── Macroeconomic Resilience Index — Country Rankings ────────────────")
print(ranking.to_string())

## ── SECTION 9: RANKING VISUALISATION

In [ ]:
fig, ax = plt.subplots(figsize=(10, 14))
colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, N))[::-1]
bars = ax.barh(ranking.index[::-1], ranking['resilience_index'][::-1],
               color=colors[::-1], edgecolor='white', linewidth=0.5)
ax.set_title('Macroeconomic Resilience Index — African Economies (PCA-Derived)',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Resilience Index (0 = most vulnerable, 100 = most resilient)')
ax.grid(True, axis='x', alpha=0.3)
ax.set_xlim(0, 115)
for bar, val in zip(bars, ranking['resilience_index'][::-1]):
    ax.text(val + 1, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}', va='center', fontsize=7)
plt.tight_layout()
plt.savefig('lecture18_demo_resilience_ranking.png', bbox_inches='tight')
plt.show()
print("Saved : lecture18_demo_resilience_ranking.png")

## ── SECTION 10: t-SNE VISUALISATION

In [ ]:
print("\nRunning t-SNE (may take ~30 seconds) ...")

tsne    = TSNE(n_components=2, random_state=SEED, perplexity=12,
               n_iter=1000, learning_rate='auto', init='pca')
tsne_coords = tsne.fit_transform(X_sc)

# Colour by resilience quartile
quartile = pd.qcut(df['resilience_index'], 4,
                   labels=['Q1 Vulnerable', 'Q2', 'Q3', 'Q4 Resilient'])
qcolors  = {'Q1 Vulnerable': '#F44336', 'Q2': '#FF9800',
            'Q3': '#4CAF50',            'Q4 Resilient': '#1565C0'}

fig, ax = plt.subplots(figsize=(12, 9))
for q, c in qcolors.items():
    mask = quartile == q
    ax.scatter(tsne_coords[mask, 0], tsne_coords[mask, 1],
               c=c, label=q, s=70, alpha=0.85, edgecolors='white', linewidths=0.5)
    for i, name in enumerate(np.array(COUNTRIES)[mask]):
        ax.annotate(name, (tsne_coords[mask, 0][i], tsne_coords[mask, 1][i]),
                    fontsize=6, ha='center', va='bottom',
                    xytext=(0, 4), textcoords='offset points')

ax.set_title('t-SNE Projection — Coloured by Resilience Quartile',
             fontsize=11, fontweight='bold')
ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.legend(title='Resilience Quartile', fontsize=8)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('lecture18_demo_tsne.png', bbox_inches='tight')
plt.show()
print("Saved : lecture18_demo_tsne.png")

## ── SECTION 11: SUMMARY

In [ ]:
print("""
╔═══════════════════════════════════════════════════════════════════╗
║  LECTURE 18 DEMO COMPLETE                                         ║
╠═══════════════════════════════════════════════════════════════════╣
║  Concepts demonstrated:                                           ║
║   1. High-dimensional macro data (12 correlated indicators)       ║
║   2. StandardScaler normalisation                                 ║
║   3. PCA — variance explained, eigenvalues, cumulative scree      ║
║   4. Loadings heatmap and factor interpretation                   ║
║   5. PCA biplot (countries + indicator arrows)                    ║
║   6. Macroeconomic Resilience Index from PC1                      ║
║   7. Country ranking bar chart                                    ║
║   8. t-SNE non-linear 2-D visualisation                           ║
╠═══════════════════════════════════════════════════════════════════╣
║  Saved outputs:                                                   ║
║   lecture18_demo_correlation.png                                  ║
║   lecture18_demo_scree.png                                        ║
║   lecture18_demo_loadings.png                                     ║
║   lecture18_demo_biplot.png                                       ║
║   lecture18_demo_resilience_ranking.png                           ║
║   lecture18_demo_tsne.png                                         ║
╚═══════════════════════════════════════════════════════════════════╝
""")